# LATINO: LAtent consisTency INverse sOlver

**Algorithm 1 from LATINO-PRO**

```
for k = 1, ..., N do
    ε ~ N(0, I)
    z_{t_k} ← √α_{t_k} · E(x^{k-1}) + √(1-α_{t_k}) · ε     ▷ Encode + noise
    u^{k}   ← D(Gθ(z_{t_k}, t_k, c))                          ▷ Denoise + decode
    x^{k}   ← prox_{δ_k · g_y}(u^{k})                         ▷ Proximal step
end for
```

## 1. Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate deepinv huggingface_hub

## 2. Imports

In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

from diffusers import (
    AutoencoderKL,
    DiffusionPipeline,
    UNet2DConditionModel,
    LCMScheduler,
)
from huggingface_hub import hf_hub_download
import deepinv as dinv
from torchvision.utils import save_image
from torchvision import transforms
from PIL import Image

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

In [ ]:
# ---- Experiment parameters (edit these) ----
IMAGE_PATH   = "sample.png"            # path to your ground-truth image
SCALE_FACTOR = 4                       # downsampling factor
N            = 4                       # LATINO iterations (4 or 8)
SIGMA_Y      = 0.05                    # observation noise std-dev
PROMPT       = "a high quality photo"  # text conditioning
OUTPUT_DIR   = "output"

# Auto-select device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Dimension must be divisible by scale_factor * 8
divisor = SCALE_FACTOR * 8
TARGET_SIZE = (1024 // divisor) * divisor
print(f"Target size: {TARGET_SIZE}x{TARGET_SIZE}  (divisible by {divisor})")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 4. Define LATINO Components

In [ ]:
# ============================================================
# Model Loading
# ============================================================

def load_pipeline(device="cuda"):
    """Load SDXL pipeline with DMD2 4-step distilled UNet and LCM scheduler."""
    vae = AutoencoderKL.from_pretrained(
        "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16
    )

    base_model_id = "stabilityai/stable-diffusion-xl-base-1.0"
    unet_config = UNet2DConditionModel.load_config(base_model_id, subfolder="unet")
    unet = UNet2DConditionModel.from_config(unet_config).to(device, torch.float16)
    unet.load_state_dict(
        torch.load(
            hf_hub_download("tianweiy/DMD2", "dmd2_sdxl_4step_unet_fp16.bin"),
            map_location=device,
            weights_only=True,
        )
    )

    pipe = DiffusionPipeline.from_pretrained(
        base_model_id,
        unet=unet,
        vae=vae,
        torch_dtype=torch.float16,
        variant="fp16",
    ).to(device)
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

    # Force VAE to float32 -- SDXL VAE is numerically unstable in float16
    pipe.vae = pipe.vae.to(dtype=torch.float32)

    return pipe


# ============================================================
# VAE Encode / Decode
# ============================================================

def vae_encode(vae, x, scaling_factor):
    """Encode image to latent space.  x in [0, 1] -> z."""
    x_norm = (2.0 * x - 1.0).to(dtype=torch.float32)
    z = vae.encode(x_norm).latent_dist.mean * scaling_factor
    return z


def vae_decode(vae, z, scaling_factor):
    """Decode latent to image space.  z -> x in [0, 1]."""
    z_f32 = z.to(dtype=torch.float32)
    x = vae.decode(z_f32 / scaling_factor).sample
    x = (x + 1.0) / 2.0
    return x.clamp(0.0, 1.0)


# ============================================================
# Diffusion Utilities
# ============================================================

def forward_diffusion(z, alpha_t):
    """Forward process: z_t = sqrt(alpha_t)*z + sqrt(1-alpha_t)*eps."""
    noise = torch.randn_like(z)
    return torch.sqrt(alpha_t) * z + torch.sqrt(1.0 - alpha_t) * noise


def consistency_denoise(pipe, z_t, timestep, prompt_embeds, pooled_prompt_embeds,
                        target_size=(1024, 1024)):
    """Single-step denoising via the consistency model (Tweedie formula)."""
    device = z_t.device
    dtype = z_t.dtype
    t = torch.tensor([timestep], device=device, dtype=torch.long)

    add_time_ids = torch.tensor(
        [[target_size[0], target_size[1], 0, 0, target_size[0], target_size[1]]],
        device=device, dtype=dtype,
    )
    added_cond_kwargs = {
        "text_embeds": pooled_prompt_embeds.to(dtype=dtype),
        "time_ids": add_time_ids,
    }

    noise_pred = pipe.unet(
        z_t, t,
        encoder_hidden_states=prompt_embeds.to(dtype=dtype),
        added_cond_kwargs=added_cond_kwargs,
    ).sample

    alpha_t = pipe.scheduler.alphas_cumprod[timestep].to(device=device, dtype=dtype)

    pred_type = pipe.scheduler.config.prediction_type
    if pred_type == "epsilon":
        z_0 = (z_t - torch.sqrt(1.0 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
    elif pred_type == "v_prediction":
        z_0 = torch.sqrt(alpha_t) * z_t - torch.sqrt(1.0 - alpha_t) * noise_pred
    elif pred_type == "sample":
        z_0 = noise_pred
    else:
        raise ValueError(f"Unknown prediction type: {pred_type}")

    return z_0


# ============================================================
# Proximal Operator  (uses deepinv's built-in prox_l2)
# ============================================================

def get_delta(t_k, df, scale_factor=4):
    """Adaptive delta_k based on timestep and measurement error (from reference)."""
    if scale_factor <= 16:
        return 1.0 * df / 10.0 if t_k > 300 else 0.5 * df / 10.0
    else:
        return 1.5 * df / 10.0 if t_k > 300 else 3.0 * df / 10.0


def get_timesteps(N):
    """Evenly spaced timesteps from 999 down (matching reference implementation)."""
    step = 1000 // N
    return [999 - i * step for i in range(N)]


# ============================================================
# Metrics
# ============================================================

def compute_psnr(pred, target):
    """Peak Signal-to-Noise Ratio (assumes [0, 1] range)."""
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return float("inf")
    return (10 * torch.log10(1.0 / mse)).item()


# ============================================================
# LATINO  --  Algorithm 1
# ============================================================

@torch.no_grad()
def latino(pipe, y, physics, prompt="a high quality photo", N=4,
           sigma_y=0.05, scale_factor=4, device="cuda", verbose=True):
    """
    LATINO: LAtent consisTency INverse sOlver.

    Args:
        pipe:         SDXL pipeline (DMD2 UNet + LCM scheduler)
        y:            Degraded observation  [B, C, H_low, W_low]  in [0, 1]
        physics:      deepinv forward operator  (e.g. Downsampling)
        prompt:       Text prompt for conditioning
        N:            Number of iterations (4 or 8)
        sigma_y:      Observation noise std-dev
        scale_factor: Downsampling factor (used for adaptive delta)
        device:       Torch device
        verbose:      Print per-iteration progress

    Returns:
        x:  Reconstructed image  [B, C, H, W]  in [0, 1]
    """
    vae = pipe.vae
    s = vae.config.scaling_factor                       # 0.13025 for SDXL
    alphas_cumprod = pipe.scheduler.alphas_cumprod.to(device)
    timesteps = get_timesteps(N)
    print(f'Time Steps: {timesteps}')

    # Proximal step operates in [-1, 1] range (matching reference calibration)
    y_norm = (y * 2 - 1).float()

    # Encode text prompt  (no classifier-free guidance for DMD2)
    prompt_embeds, _, pooled_prompt_embeds, _ = pipe.encode_prompt(
        prompt, device=device, num_images_per_prompt=1,
        do_classifier_free_guidance=False,
    )

    # Infer target image size from observation + scale factor
    B, C, H_low, W_low = y.shape
    H, W = H_low * scale_factor, W_low * scale_factor

    # x^(0): initialise with bicubic upsampling of y
    x = F.interpolate(y, size=(H, W), mode="bicubic", align_corners=False)
    x = x.clamp(0.0, 1.0)

    pbar = tqdm(enumerate(timesteps), total=len(timesteps),
                desc="LATINO") if verbose else enumerate(timesteps)

    for k, t_k in pbar:
        # 1) Encode:  z = E(x^{k-1})
        z = vae_encode(vae, x, s)

        # 2) Forward diffusion:  z_{t_k} = sqrt(alpha)*z + sqrt(1-alpha)*eps
        alpha_t = alphas_cumprod[t_k].to(z.device, z.dtype)
        z_t = forward_diffusion(z, alpha_t)

        # Cast to UNet dtype (float16 on CUDA) for the denoising step
        z_t = z_t.to(dtype=pipe.unet.dtype)

        # 3) Denoise:  z_0 = G_theta(z_{t_k}, t_k, c)
        z_0 = consistency_denoise(
            pipe, z_t, t_k, prompt_embeds, pooled_prompt_embeds,
            target_size=(H, W),
        )

        # 4) Decode:  u^{k} = D(z_0)
        u = vae_decode(vae, z_0, s)

        # 5) Proximal step:  x^{k} = prox_{delta_k * g_y}(u^{k})
        #    Operate in [-1, 1] range; use delta_k directly as gamma
        u_norm = (u * 2 - 1).float()

        df = torch.norm(physics.A(u_norm) - y_norm).item()
        delta_k = get_delta(t_k, df, scale_factor=scale_factor)

        prox_x_norm = physics.prox_l2(u_norm, y=y_norm, gamma=delta_k)
        x = ((prox_x_norm + 1) / 2).clamp(0.0, 1.0)

        if verbose:
            pbar.set_postfix(t=t_k, gamma=f"{delta_k:.4f}", df=f"{df:.4f}")

    return x


print("All components defined.")

## 5. Download / Load Image

In [ ]:
# Option A: Download a sample CelebA-HQ face
# (uncomment these lines if you don't have a local image)

!pip install -q datasets
from datasets import load_dataset
ds = load_dataset("mattymchen/celeba-hq", split="train", streaming=True)
sample = next(iter(ds))
sample["image"].save("sample.png")
IMAGE_PATH = "sample.png"

# Option B: Upload your own image in Colab
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = list(uploaded.keys())[0]

# Load and prepare
# image = Image.open(IMAGE_PATH).convert("RGB")
image = sample["image"].convert("RGB")
transform = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.CenterCrop((TARGET_SIZE, TARGET_SIZE)),
    transforms.ToTensor(),
])
gt = transform(image).unsqueeze(0).to(device)
print(f"Ground truth shape: {gt.shape}")

## 6. Create Degraded Observation

In [ ]:
physics = dinv.physics.Downsampling(
    img_size=(3, TARGET_SIZE, TARGET_SIZE),
    factor=SCALE_FACTOR,
    filter="bicubic",
    device=device,
)

y = physics.A(gt)
if SIGMA_Y > 0:
    y = y + SIGMA_Y * torch.randn_like(y)
    y = y.clamp(0.0, 1.0)

print(f"Observation y shape: {y.shape}")

# Bicubic baseline
y_up = F.interpolate(y, size=(TARGET_SIZE, TARGET_SIZE),
                     mode="bicubic", align_corners=False).clamp(0, 1)
baseline_psnr = compute_psnr(y_up, gt)
print(f"Bicubic baseline PSNR: {baseline_psnr:.2f} dB")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(gt[0].cpu().permute(1, 2, 0))
axes[0].set_title("Ground Truth")
axes[0].axis("off")
axes[1].imshow(y_up[0].cpu().permute(1, 2, 0))
axes[1].set_title(f"Degraded (bicubic up) -- PSNR {baseline_psnr:.2f} dB")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 7. Load SDXL + DMD2 Pipeline

In [ ]:
pipe = load_pipeline(device=device)
print("Pipeline loaded.")
print(f"  Scheduler: {pipe.scheduler.__class__.__name__}")
print(f"  Prediction type: {pipe.scheduler.config.prediction_type}")
print(f"  VAE dtype: {pipe.vae.dtype}")
print(f"  UNet dtype: {pipe.unet.dtype}")

## 8. Run LATINO

In [ ]:
N = 8
result = latino(
    pipe, y, physics,
    prompt=PROMPT,
    N=N,
    sigma_y=SIGMA_Y,
    scale_factor=SCALE_FACTOR,
    device=device,
    verbose=True,
)

latino_psnr = compute_psnr(result, gt)
print(f"\nLATINO PSNR: {latino_psnr:.2f} dB  (baseline: {baseline_psnr:.2f} dB)")

## 9. Visualise Results

In [ ]:
gt_img = gt[0].cpu().permute(1, 2, 0)
yup_img = y_up[0].cpu().permute(1, 2, 0).clamp(0, 1)
res_img = result[0].cpu().permute(1, 2, 0).clamp(0, 1)

# Absolute error maps (amplified for visibility)
amp = 5
diff_gt = (gt_img - gt_img).abs() * amp          # zero (reference)
diff_yup = (yup_img - gt_img).abs() * amp
diff_res = (res_img - gt_img).abs() * amp

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: images
axes[0, 0].imshow(gt_img)
axes[0, 0].set_title("Ground Truth")
axes[0, 0].axis("off")

axes[0, 1].imshow(yup_img)
axes[0, 1].set_title(f"Degraded (bicubic) -- {baseline_psnr:.2f} dB")
axes[0, 1].axis("off")

axes[0, 2].imshow(res_img)
axes[0, 2].set_title(f"LATINO (N={N}) -- {latino_psnr:.2f} dB")
axes[0, 2].axis("off")

# Row 2: |image - GT| x amp
axes[1, 0].imshow(diff_gt.clamp(0, 1))
axes[1, 0].set_title(f"|GT - GT| x{amp}")
axes[1, 0].axis("off")

axes[1, 1].imshow(diff_yup.clamp(0, 1))
axes[1, 1].set_title(f"|Degraded - GT| x{amp}")
axes[1, 1].axis("off")

axes[1, 2].imshow(diff_res.clamp(0, 1))
axes[1, 2].set_title(f"|LATINO - GT| x{amp}")
axes[1, 2].axis("off")

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved comparison to {OUTPUT_DIR}/comparison.png")

## 10. Save Outputs

In [ ]:
out = Path(OUTPUT_DIR)
save_image(gt, out / "ground_truth.png")
save_image(y_up.clamp(0, 1), out / "degraded_bicubic.png")
save_image(result.clamp(0, 1), out / "latino_result.png")
print(f"Images saved to {out}/")